In [118]:
from collections import Counter
import numpy as np
import pandas as pd
import os

In [107]:
# Define paths
# root = os.path.dirname(os.path.abspath(__file__))
root = "/home/acomajuncosa/Documents/mtb-targeted-protein-degradation/scripts"
data_path = os.path.abspath(os.path.join(root, "..", "data", "sequences", "interpro"))
outdir = os.path.abspath(os.path.join(root, "..", "processed", "sequences"))

# Define uniprots
uniprots = [i.split(".")[0].split("-")[2] for i in sorted(os.listdir(data_path))]
uniprots_to_number = {i: c for c,i in enumerate(sorted(uniprots))}

dfs = []

for uni in uniprots:

    # Load TSV file
    df = pd.read_csv(os.path.join(data_path, f"entry-matching-{uni}.tsv"), sep="\t")
    
    # Select only interpro data
    df = df[df['Source Database'] == 'interpro']

    # Select only relevant columns
    df = df[['Accession', 'Name', "Source Database", 'Type', 'Protein Accession', 'Protein Length', 'Matches']]
    df['Protein Accession'] = df['Protein Accession'].str.upper()

    # Append to dfs
    dfs.append(df)

# Merge everything
dfs = pd.concat(dfs, ignore_index=True)

# Create dict - from accession to name and type
acc_to_name_type = {i: [j, k] for i,j,k in zip(dfs['Accession'], dfs['Name'], dfs['Type'])}

In [148]:
interpro_data = []
interpro_data_collapsed = []

for interpro_id in sorted(acc_to_name_type):

    # Store data
    name, type_ = acc_to_name_type[interpro_id]
    proteins = ",".join(dfs[dfs['Accession'] == interpro_id]['Protein Accession'].tolist())

    # Create binary vector
    vec = np.zeros(len(uniprots), dtype=int)
    ind = [uniprots_to_number[i] for i in sorted(uniprots_to_number) if i in proteins]
    vec[ind] = 1

    # Append to final df
    cols = [interpro_id, name, type_, len(proteins.split(",")), proteins]
    cols.extend(vec)
    interpro_data.append(cols)

# Create dataframe
interpro_data = pd.DataFrame(interpro_data, columns=['Accession', 'Name', 'Type', 'Number of proteins', 'Proteins'] + sorted(uniprots))
interpro_data = interpro_data.sort_values('Number of proteins', ascending=False).reset_index(drop=True)

# Save results
interpro_data.to_csv(os.path.join(outdir, 'interpro_summary.tsv'), sep='\t', index=False)

In [149]:
interpro_data

,Accession,Name,Type,Number of proteins,Proteins,P9WFS9,P9WFT1,P9WFT3,P9WFT5,P9WFT7,...,P9WFV3,P9WFV5,P9WFV7,P9WFV9,P9WFW1,P9WFW3,P9WFW5,P9WFW7,P9WN61,P9WQA1
0,IPR045864,Class II Aminoacyl-tRNA synthetase/Biotinyl pr...,homologous_superfamily,10,"P9WFT5,P9WFT7,P9WFT9,P9WFU1,P9WFU3,P9WFU9,P9WF...",0,0,0,1,1,...,0,1,1,0,0,1,0,1,0,0
1,IPR014729,Rossmann-like alpha/beta/alpha sandwich fold,homologous_superfamily,9,"P9WFS9,P9WFT1,P9WFT3,P9WFU5,P9WFV1,P9WFV3,P9WF...",1,1,1,0,0,...,1,0,0,1,1,0,1,0,0,0
2,IPR006195,"Aminoacyl-tRNA synthetase, class II",domain,8,"P9WFT5,P9WFT7,P9WFT9,P9WFU3,P9WFU9,P9WFV5,P9WF...",0,0,0,1,1,...,0,1,1,0,0,1,0,0,0,0
3,IPR001412,"Aminoacyl-tRNA synthetase, class I, conserved ...",conserved_site,7,"P9WFS9,P9WFT1,P9WFT3,P9WFU5,P9WFV1,P9WFV3,P9WFW5",1,1,1,0,0,...,1,0,0,0,0,0,1,0,0,0
4,IPR009080,"Aminoacyl-tRNA synthetase, class Ia, anticodon...",homologous_superfamily,6,"P9WFS9,P9WFU5,P9WFV1,P9WFV3,P9WFW1,P9WFW5",1,0,0,0,0,...,1,0,0,0,1,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,IPR049940,Glutamyl-Q tRNA(Asp) synthetase/Glutamate--tRN...,family,1,P9WFV9,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
125,IPR050058,Alanine--tRNA ligase,family,1,P9WFW7,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
126,IPR050062,Proline-tRNA synthetase,family,1,P9WFT9,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127,IPR050203,Tryptophan--tRNA ligase,family,1,P9WFT3,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [104]:
dfs[dfs['Accession'] == 'IPR000120']

,Accession,Name,Source Database,Type,Protein Accession,Protein Length,Matches
183,IPR000120,Amidase,interpro,family,P9WQA1,494,3..485
